# Clean `emr_cycle_aspirations`

A first-pass cleaning of the `emr_cycle_aspirations` table (3228 rows, 40 columns) before analysis.

What this notebook does, in order:
1. **Profile** every column (how full it is, how many distinct values, its type) and delete non-informative columns
2. **Keep** selected subset of columns
3. **Save** a cleaned copy

The same `profile()` function works on the other `emr_*` tables too, so you can reuse this pattern.


> **Important:** keep your raw patient data in a `data/` folder that is git-ignored. Don't commit EMR exports to GitHub, even a private repo.

## Setup

In [17]:
from pathlib import Path
import pandas as pd
import import_ipynb

import helper_functions

In [18]:
# --- point this at your raw export ---
PATH = Path("../data/emr_cycle_aspirations.csv")

# loading data
df = pd.read_csv(PATH)


In [19]:
#setting display options to show all columns and full width
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)        # Allow full width

In [20]:
print(df.shape)

(3228, 40)


## 1. Profile every column

One row per column so you can see at a glance what's worth keeping.

In [21]:
# now profiling the dataframes to get a better understanding of the data and dropping unninformative columns

prof = helper_functions.profile(df)
df = helper_functions.drop_empty_and_constant_columns(df, prof)
print(prof)

Dropping 15 columns
                            non_null  nulls  distinct    dtype  pct_null  \
id                              3228      0      3228      str       0.0   
addedby                         3228      0        14    int64       0.0   
addedon                         3228      0      3228      str       0.0   
cycleid                         3228      0      3220      str       0.0   
patid                           3228      0      2404    int64       0.0   
for_position                    3228      0         1      str       0.0   
technique                       3228      0         1      str       0.0   
retrieved                       3228      0      3220      str       0.0   
denuding_type                   3228      0         1      str       0.0   
oocytes_retrieved               3227      1        52  float64       0.0   
retrievedby                     3217     11        11  float64       0.3   
retrievedtech                   3216     12        11  float64      

## 2. Keeping subset of columns

Only looking to keep columns that inform response type. Filtering beyond the NA and single value columns

In [22]:
# print list of columns with their data types and percentage of nulls
print("\nColumn summary:")
for col in df.columns:
    non_null = df[col].notna().sum()
    total = len(df)
    pct_null = (total - non_null) / total * 100
    dtype = df[col].dtype
    print(f"{col}: {dtype}, {pct_null:.1f}% null")


Column summary:
id: str, 0.0% null
addedby: int64, 0.0% null
addedon: str, 0.0% null
deletedby: float64, 99.7% null
deletedon: str, 99.7% null
cycleid: str, 0.0% null
patid: int64, 0.0% null
retrieved: str, 0.0% null
retrievedby: float64, 0.3% null
oocytes_retrieved: float64, 0.0% null
complication: str, 99.9% null
note: str, 72.6% null
denuding_performed: str, 1.2% null
denuding_performedby: float64, 1.6% null
patient_identifiedby: float64, 3.8% null
retrievedtech: float64, 0.4% null
spermwitness: float64, 99.8% null
labassistant: float64, 0.7% null
scrub_tech: float64, 3.2% null
crna: str, 1.1% null
needle_type: str, 1.0% null
duration: str, 1.6% null
accession_number: str, 1.3% null
facility: float64, 12.3% null
patient_identifiedby_other: str, 97.5% null


In [23]:
# create a list of columns that I want to keep
keep_cols = [
    "id", "cycleid", "patid", "retrieved", "retrievedby", "oocytes_retrieved", 
    "complication", "note", "denuding_performed", "denuding_performedby", "retrievedtech", 
    "labassistant", "scrub_tech", "crna", "needle_type", "duration", "accession_number" 
]
# filter the dataframe to keep only those columns
df = df[keep_cols]
print(f"\nAfter filtering to {len(keep_cols)} columns: {df.shape[1]} columns")


After filtering to 17 columns: 17 columns


In [24]:
print(df.shape)

(3228, 17)


## 3. Cleaning data

In [25]:
# Cleaning needle type

# Make needle_type lowercase and remove extra spaces
df["needle_type"] = (
    df["needle_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# now applying my helper function
df["needle_type_cleaned"] = (df["needle_type"].apply(helper_functions.clean_needle_type))

## 4. Saving the df

In [26]:
# save as csv
df.to_csv("../data/emr_cycle_aspirations_processed.csv", index=False)